[![img/cloudevel.png](img/cloudevel.png)](https://cloudevel.com)

# Consultas básicas.

**sql101 · SQL para Ingeniería y Analítica de Datos** · Cloudevel® · Instructor: José Luis Chiquete Valdivieso · Módulo 1: *Fundamentos*

> 📝 **Versión del alumno.** Las celdas marcadas con ```-- TU CÓDIGO AQUÍ``` y las de *Tu respuesta* son para ti.

En este *notebook* se escribirán las primeras consultas: selección de columnas, filtros, valores nulos, ordenamiento y límite de resultados.

## 🧭 Cómo usar este *notebook*.

* Ejecuta primero la celda de **configuración** y después las demás **en orden** (```Shift + Enter```).
* Las celdas que comienzan con ```%%sql``` son consultas: todo lo que sigue a ```%%sql``` es *SQL* y puede copiarse a otro motor.
* **Ejercicio:** lo que todos deben dominar. **Reto (opcional):** para quien quiere profundizar; se puede resolver como práctica.
* En las celdas ```-- TU CÓDIGO AQUÍ```, reemplaza la línea ```SELECT NULL AS tu_respuesta;``` por tu consulta. Las de *✍️ Tu respuesta* son para escribir.

## Configuración.

Abre la base de datos del taller en modo de sólo lectura y habilita las celdas ```%%sql```.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/thelook.duckdb', read_only=True)
%sql conexion

Las consultas se escriben con el lenguaje de consulta de datos (*DQL*), cuya instrucción central es ```SELECT```.

Muchas consultas de esta lección terminan con ```ORDER BY ... FETCH FIRST 20 ROWS ONLY```: ordenan el resultado y regresan sólo las primeras 20 filas, para que sea legible. Ambas cláusulas se explican al final de la lección.

## ```SELECT``` y ```FROM```.

* La cláusula ```SELECT``` indica qué columnas o expresiones se regresan.
* La cláusula ```FROM``` indica de qué tabla se leen los datos.

In [ ]:
%%sql
SELECT name, brand, category, retail_price
FROM products
ORDER BY name
FETCH FIRST 20 ROWS ONLY;

### Alias con ```AS```.

Un alias renombra una columna en el resultado. Si el alias contiene espacios o caracteres especiales, se delimita con **comillas dobles**, que en el estándar se usan para identificadores (las comillas simples son para texto).

In [ ]:
%%sql
SELECT name AS producto,
  brand AS marca,
  retail_price AS "precio de lista"
FROM products
ORDER BY producto
FETCH FIRST 20 ROWS ONLY;

> **Nota de dialecto:** en *BigQuery* y *MySQL* las comillas dobles delimitan texto, y los identificadores se delimitan con acentos graves (```` `precio de lista` ````).

### Columnas calculadas.

Es posible operar sobre las columnas y aplicar funciones. La siguiente consulta calcula el margen de cada producto y lo redondea con ```ROUND()```.

In [ ]:
%%sql
SELECT name,
  retail_price,
  cost,
  ROUND(retail_price - cost, 2) AS margen,
  ROUND((retail_price - cost) / retail_price * 100, 1) AS margen_pct
FROM products
ORDER BY margen DESC
FETCH FIRST 20 ROWS ONLY;

### ```SELECT *```.

```SELECT *``` regresa todas las columnas. Es útil para explorar, pero en consultas definitivas conviene nombrar las columnas:

* El resultado no cambia si alguien agrega columnas a la tabla.
* **En los motores columnares** (*DuckDB*, *BigQuery*, *Snowflake*) **cada columna se almacena por separado, y el motor sólo lee las columnas que la consulta menciona.** Seleccionar menos columnas reduce el trabajo; en *BigQuery*, además, reduce directamente el costo.

In [ ]:
%%sql
SELECT *
FROM distribution_centers;

## ```DISTINCT```.

```DISTINCT``` elimina filas duplicadas del resultado.

In [ ]:
%%sql
SELECT DISTINCT category
FROM products
ORDER BY category;

In [ ]:
%%sql
SELECT DISTINCT department, category
FROM products
ORDER BY department, category;

## Filtrado con ```WHERE```.

La cláusula ```WHERE``` conserva sólo las filas que cumplen una condición.

| Operador | Significado |
|---|---|
| ```=```, ```<>``` | Igual, distinto (```!=``` también es común, pero ```<>``` es el estándar) |
| ```<```, ```<=```, ```>```, ```>=``` | Comparaciones |
| ```AND```, ```OR```, ```NOT``` | Operadores lógicos |
| ```IN (...)``` | Pertenece a una lista |
| ```BETWEEN a AND b``` | Dentro de un rango (incluye ambos extremos) |
| ```LIKE``` | Coincide con un patrón de texto |
| ```IS NULL```, ```IS NOT NULL``` | Ausencia o presencia de valor |

* Productos con precio mayor a 500.

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
WHERE retail_price > 500
ORDER BY retail_price DESC;

* Productos de la categoría ```Jeans``` del departamento ```Women```.

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
WHERE category = 'Jeans' AND department = 'Women'
ORDER BY retail_price;

### Precedencia de ```AND``` y ```OR```.

```AND``` se evalúa antes que ```OR```. Compara estas dos consultas, ordenadas del precio más alto al más bajo: la primera regresa todos los ```Jeans``` sin importar el precio, más los ```Pants``` baratos.

In [ ]:
%%sql
SELECT name, category, retail_price
FROM products
WHERE category = 'Jeans' OR category = 'Pants' AND retail_price < 30
ORDER BY retail_price DESC
FETCH FIRST 20 ROWS ONLY;

In [ ]:
%%sql
SELECT name, category, retail_price
FROM products
WHERE (category = 'Jeans' OR category = 'Pants')
  AND retail_price < 30
ORDER BY retail_price DESC
FETCH FIRST 20 ROWS ONLY;

### ```IN``` y ```BETWEEN```.

In [ ]:
%%sql
SELECT name, category, retail_price
FROM products
WHERE category IN ('Jeans', 'Pants', 'Shorts')
  AND retail_price BETWEEN 20 AND 40
ORDER BY retail_price;

### ```LIKE```.

* ```%``` representa cualquier secuencia de caracteres (incluso vacía).
* ```_``` representa exactamente un carácter.
* ```LIKE``` distingue mayúsculas y minúsculas; para evitarlo se compara contra ```LOWER(columna)```.

In [ ]:
%%sql
SELECT name, brand
FROM products
WHERE LOWER(name) LIKE '%denim%'
ORDER BY name
FETCH FIRST 20 ROWS ONLY;

> **Nota de dialecto:** *DuckDB* y *PostgreSQL* ofrecen ```ILIKE``` para comparar sin distinguir mayúsculas; no es estándar. En *SQLite* y *MySQL*, ```LIKE``` no distingue mayúsculas por omisión.

### Valores ```NULL```.

```NULL``` representa la ausencia de un valor. **Cualquier comparación con ```NULL``` usando ```=``` da como resultado ```NULL``` (desconocido), no ```TRUE```**, por lo que se deben usar ```IS NULL``` e ```IS NOT NULL```.

* Esta consulta no regresa nada, aunque hay productos sin marca:

In [ ]:
%%sql
SELECT id, name, brand
FROM products
WHERE brand = NULL;

* Esta sí:

In [ ]:
%%sql
SELECT id, name, brand
FROM products
WHERE brand IS NULL;

* ```COALESCE()``` regresa el primer valor no nulo de una lista; es útil para reemplazar ```NULL``` por un valor por omisión.

In [ ]:
%%sql
SELECT id, name, COALESCE(brand, 'Sin marca') AS marca
FROM products
WHERE brand IS NULL;

## Ordenamiento con ```ORDER BY```.

* ```ORDER BY``` ordena el resultado de forma ascendente (```ASC```, por omisión) o descendente (```DESC```).
* Se pueden indicar varias columnas: la segunda desempata a la primera.
* **Sin ```ORDER BY```, el orden de las filas no está garantizado**, aunque parezca estable.

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
ORDER BY retail_price DESC, name
FETCH FIRST 20 ROWS ONLY;

## Limitar el resultado: ```FETCH FIRST``` y ```LIMIT```.

La forma estándar de obtener sólo las primeras ```n``` filas es ```FETCH FIRST n ROWS ONLY```. Combinada con ```ORDER BY```, responde preguntas de tipo *"los n mayores"*.

* Los 10 productos más caros.

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
ORDER BY retail_price DESC, name
FETCH FIRST 10 ROWS ONLY;

> **Nota de dialecto:** ```LIMIT n``` no es estándar, pero es la forma más difundida (*DuckDB*, *PostgreSQL*, *MySQL*, *SQLite*, *BigQuery*, *Snowflake*). *BigQuery* y *SQLite* **sólo** admiten ```LIMIT```; *SQL Server* admite ```TOP n``` y, desde *SQL Server* 2012, ```ORDER BY … OFFSET 0 ROWS FETCH NEXT n ROWS ONLY```, la forma más cercana a ```FETCH FIRST``` (este último, en *SQL Server*, exige ```OFFSET```).

In [ ]:
%%sql
SELECT name, brand, retail_price
FROM products
ORDER BY retail_price DESC, name
LIMIT 10;

### Limitar el resultado no limita la búsqueda.

```FETCH FIRST``` y ```LIMIT``` se aplican **al final** del procesamiento: recortan el resultado que se entrega, no la búsqueda.

* Con ```ORDER BY```, ```GROUP BY``` o ```DISTINCT```, el motor debe procesar todas las filas para saber cuáles son las primeras.
* En *BigQuery*, ```LIMIT``` **no reduce los bytes procesados ni el costo**.
* Lo que realmente reduce el trabajo es **seleccionar menos columnas** y **filtrar con ```WHERE```**.

En este taller, las consultas que regresan muchas filas terminan con ```FETCH FIRST``` **sólo para que el resultado sea legible**, siempre acompañado de ```ORDER BY``` para que las filas mostradas sean predecibles.

## Orden lógico de ejecución.

Aunque se escribe primero ```SELECT```, el motor evalúa las cláusulas en este orden:

```
FROM → WHERE → SELECT → ORDER BY → FETCH FIRST / LIMIT
```

Por esta razón, **un alias definido en ```SELECT``` no puede usarse en ```WHERE```**, pero sí en ```ORDER BY```.

In [ ]:
%%sql
SELECT name, ROUND(retail_price - cost, 2) AS margen
FROM products
WHERE retail_price - cost > 300   -- aquí no se puede usar "margen"
ORDER BY margen DESC;

> **Nota de dialecto:** *DuckDB* permite, como extensión, usar alias en ```WHERE```. No lo hagas: la consulta fallaría en otros motores.

## ✏️ Ejercicios.

Los **ejercicios** cubren lo que todos deben dominar; los **retos** son opcionales. Las soluciones están en ```soluciones/```.

### Ejercicio 1 · Productos baratos.

Obtén el nombre, la marca y el precio de los productos del departamento ```Women``` con precio menor a 10, ordenados del más barato al más caro.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 2 · Países distintos.

Lista los distintos países (```country```) de la tabla ```users```, en orden alfabético.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 3 · Órdenes canceladas o devueltas.

Obtén las órdenes (```order_id```, ```user_id```, ```status```) cuyo estado sea ```Cancelled``` o ```Returned```. Muestra las primeras 20 por ```order_id```.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 4 · Patrones y columnas calculadas.

Encuentra los productos cuyo nombre contenga la palabra ```hoodie``` (sin importar mayúsculas) y cuyo margen sea mayor a 50. Muestra nombre, precio, costo y margen.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Reto 1 · Los clientes de mayor edad.

Obtén los 5 clientes de mayor edad de ```Brasil``` (así se escribe en este conjunto de datos): nombre, apellido, edad y estado. En caso de empate en la edad, ordena por apellido.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ (opcional): reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>